# Notebook 2: Web Scraping SpaceX Launch Records with BeautifulSoup
## IBM Data Science Professional Certificate Capstone Project

### Learning Objectives:
1. Understand the basics of HTML tables and the Document Object Model (DOM).
2. Use Python's `BeautifulSoup` library to parse web pages.
3. Extract tabular flight information, dates, booster versions, and launch outcomes.
4. Convert scraped web data into a structured Pandas DataFrame.

---
### 1. What is Web Scraping?
- **Web Scraping:** The automated extraction of unstructured or semi-structured data from web pages (HTML) into structured tabular records.
- We target Wikipedia's public Falcon 9 launch history page:
  `https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches`

In [ ]:
# Step 1: Import web scraping libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os

print("Web scraping packages loaded successfully!")

### 2. Request and Parse HTML Page
We send an HTTP GET request with a standard browser User-Agent header and feed the HTML response into BeautifulSoup.

In [ ]:
# Step 2: Fetch Wikipedia page HTML
WIKI_URL = "https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

try:
    response = requests.get(WIKI_URL, headers=headers, timeout=15)
    soup = BeautifulSoup(response.text, "html.parser")
    print(f"Page Title: {soup.title.string}")
except Exception as e:
    print(f"Notice: {e}")
    soup = None

### 3. Find Launch Tables and Parse Column Headers
Wikipedia organizes flight records inside HTML `<table>` tags with the CSS class `wikitable`.

In [ ]:
# Step 3: Extract tables and table rows
if soup:
    tables = soup.find_all("table", class_="wikitable")
    print(f"Found {len(tables)} launch tables on the page.")
    
    extracted_rows = []
    for table in tables:
        rows = table.find_all("tr")
        for row in rows:
            cols = row.find_all(["th", "td"])
            if len(cols) >= 6:
                row_data = [col.get_text(strip=True) for col in cols]
                first_val = row_data[0].replace("[", "").replace("]", "")
                if first_val.isdigit():
                    extracted_rows.append({
                        "FlightNumber": first_val,
                        "Date": row_data[1] if len(row_data) > 1 else "",
                        "BoosterVersion": row_data[2] if len(row_data) > 2 else "Falcon 9",
                        "LaunchSite": row_data[3] if len(row_data) > 3 else "",
                        "Payload": row_data[4] if len(row_data) > 4 else "",
                        "PayloadMass": row_data[5] if len(row_data) > 5 else "",
                        "Orbit": row_data[6] if len(row_data) > 6 else "",
                        "Outcome": row_data[7] if len(row_data) > 7 else ""
                    })
    df_scraped = pd.DataFrame(extracted_rows)
    print(f"Parsed {len(df_scraped)} launch records from Wikipedia HTML.")
else:
    # Verified fallback
    fallback_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_web_scraped.csv"
    df_scraped = pd.read_csv(fallback_url)
    print(f"Loaded verified scraped dataset: {df_scraped.shape}")

df_scraped.head(5)

### 4. Save Scraped Launch Records

In [ ]:
# Step 4: Export to data/raw/
os.makedirs("../data/raw", exist_ok=True)
scraped_out = "../data/raw/spacex_web_scraped.csv"
df_scraped.to_csv(scraped_out, index=False)
print(f"Web scraped records saved to: {scraped_out}")

---
### Practice Exercise & Check for Understanding:
1. What does the HTML tag `<tr>` represent? *(Answer: Table Row)*
2. What does `find_all('table', class_='wikitable')` do? *(Answer: Finds all table elements having the CSS styling class 'wikitable')*.

### Summary & Next Steps:
We now possess both API and web-scraped launch records. In Notebook 3, we proceed to Data Wrangling to clean missing values and create our landing success target variable!